# 新能源汽车经营分析

## 项目概述
这是一个面向实习面试的新能源汽车销售数据分析项目。我会使用中国新能源汽车及整体汽车的月度销量数据，跟踪销量、同比、环比、市场份额、品牌排名和车型结构。

项目希望练习从原始数据整理、指标计算、图表展示到业务解读的完整过程。分析中会把“数据观察”“可能原因”和“待验证假设”分开写，避免把销量变化直接当作因果结论。

### 数据范围（读取后补充）
| 项目 | 内容 |
|---|---|
| 数据起止月份 | 预计为2015年至2023年11月，待读取确认 |
| 更新周期 | 月度 |
| 统计粒度 | 厂商/品牌/车型/月，待读取确认 |
| 数据来源 | 项目目录中的Excel数据及说明文件 |
| 最新更新时间 | 待填写 |

> 如果2023年只覆盖1—11月，年度比较必须采用同周期口径，不与完整年度直接比较。

### AI使用情况
- 使用AI辅助梳理分析框架和检查代码思路。
- 后续如使用AI辅助代码编写、纠错或文案整理，会在此补充说明。

# 经营监控目标

1. 了解新能源汽车销量、同比/环比、市场份额、品牌排名和车型结构的月度变化。
2. 用清晰的阈值找出销量、份额或排名变化较大的月份。
3. 练习区分数据事实、可能解释和还需要验证的假设。
4. 尝试使用Plotly制作可筛选的Notebook内看板。
5. 最后整理一页简洁的月度摘要，方便复盘重点变化。

---
# 初始化与运行参数

运行前请确保已安装：`pandas`、`numpy`、`plotly`。

In [1]:
# 数据处理
import pandas as pd
import numpy as np
import duckdb

# 文件与路径管理
from pathlib import Path
import openpyxl

# 可视化
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# 辅助工具
import warnings

# Plotly 默认主题
pio.templates.default = "plotly_white"

---
# 指标口径与异常规则

## 核心指标

| 指标 | 本项目口径 |
|---|---|
| 月销量 | 对象在自然月内的销量合计 |
| 同比增长率 | 当月销量相对上年同月销量的变化率 |
| 环比增长率 | 当月销量相对上月销量的变化率 |
| 品牌市场份额 | 品牌当月销量 ÷ 同期新能源汽车市场总销量 |
| 品牌销量排名 | 按同月品牌销量降序排列 |
| 排名变化 | 当月排名与上月排名之差 |
| 车型销量贡献率 | 车型当月销量 ÷ 所属品牌当月销量 |
| TOP1/TOP3集中度 | 品牌销量最高的1/3款车型销量之和 ÷ 品牌当月销量 |

## 默认异常规则

- 销量同比绝对变化达到30%。
- 销量环比绝对变化达到20%。
- 市场份额单月绝对变化达到2个百分点。
- 品牌排名单月变化达到3位。
- 低于最低销量基数的对象不直接标记为经营异常，单列为“低基数波动”。
- 新进入或上月无销量的对象标记为“新增记录”，不计算普通环比。
- 同比基期无销量时不计算同比，避免除零或误导。

> 最低销量基数会在查看数据分布后再确定。本项目先使用这些规则进行初步筛选，异常结果仍需要结合具体品牌和车型进一步判断。

---
# 数据导入与数据理解

先读取新能源汽车、整体汽车和总体销量数据，检查工作表、字段、统计粒度、时间范围和销量单位。若数据中没有独立品牌字段，暂时使用“厂商”作为品牌分析维度，并在后续说明中记录这一限制。

In [2]:
# TODO：设置Excel文件路径
sheet0 = '中国汽车总体销量.xlsx'
sheet1 = '中国汽车每月销售表.xlsx'
sheet2 = '中国电动车每月销售表.xlsx'
excel_file = [sheet0, sheet1, sheet2]

db = 'Chinese EV Sales Database.db'

In [3]:
# 查看excel文件的字段
for sheet_name in excel_file:
    print(f"Sheet: {sheet_name}")
    df = pd.read_excel(sheet_name)
    print(df.head())
    print("\n")

Sheet: 中国汽车总体销量.xlsx
          时间       销量      同比
0 2023-09-01  2019445 -0.1125
1 2023-08-01  1922495 -0.1339
2 2023-07-01  1781580 -0.1620
3 2023-06-01  1894250 -0.1254
4 2023-05-01  1685966  0.0691


Sheet: 中国汽车每月销售表.xlsx
     年份  月份  排名           车型      厂商     销量       售价（万元）
0  2022   1   1           轩逸    东风日产  61170   9.98-17.49
1  2022   1   2           朗逸    上汽大众  45524   9.40-15.19
2  2022   1   3     宏光MINIEV  上汽通用五菱  37048    3.28-9.99
3  2022   1   4         哈弗H6    长城汽车  35570   9.89-15.70
4  2022   1   5  长安CS75 PLUS    长安汽车  33590  11.79-15.49


Sheet: 中国电动车每月销售表.xlsx
     年份  月份  排名      厂商        车型     销量         售价（万元）
0  2023   1   1     比亚迪  宋PLUS新能源  35585  15.48 - 21.99
1  2023   1   2     比亚迪        海豚  17582  11.68 - 13.98
2  2023   1   3  上汽通用五菱  宏光MINIEV  16416    3.28 - 9.99
3  2023   1   4     比亚迪     元PLUS  14342  13.58 - 16.78
4  2023   1   5   特斯拉中国   Model Y  14184  25.99 - 35.99




In [7]:
# TODO：读取新能源汽车、整体汽车和总体销量数据
_ = duckdb.connect(db)

_.execute(f"""
CREATE OR REPLACE TABLE Total_Sales AS
    SELECT
        时间 AS time,
        销量 AS total_sales,
        同比 AS yoy_growth      
FROM read_xlsx({sheet0})
""")

_.execute(f"""
CREATE OR REPLACE TABLE V_Sales AS
    SELECT
        年份 AS year,
        月份 AS month,
        车型 AS model,
        厂商 AS manufacturer,
        销量 AS sales,
        售价（万元） AS price
FROM read_xlsx({sheet1})
""")

_.execute(f"""
CREATE OR REPLACE TABLE EV_Sales AS
    SELECT
        年份 AS year,
        月份 AS month,
        车型 AS model,
        厂商 AS manufacturer,
        销量 AS sales,
        售价（万元） AS price
FROM read_xlsx({sheet2})
""")

_.close()

In [15]:
# 构建dataframe
ts_df = pd.read_sql("SELECT * FROM Total_Sales", duckdb.connect(db))
v_df = pd.read_sql("SELECT * FROM V_Sales", duckdb.connect(db))
ev_df = pd.read_sql("SELECT * FROM EV_Sales", duckdb.connect(db))

dfs = [ts_df, v_df, ev_df]

C:\Users\13485\AppData\Local\Temp\ipykernel_29488\819704523.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  ts_df = pd.read_sql("SELECT * FROM Total_Sales", duckdb.connect(db))
C:\Users\13485\AppData\Local\Temp\ipykernel_29488\819704523.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  v_df = pd.read_sql("SELECT * FROM V_Sales", duckdb.connect(db))
C:\Users\13485\AppData\Local\Temp\ipykernel_29488\819704523.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  ev_df = pd.read_sql("SELECT * FROM EV_Sales", duckdb.connect

In [17]:
# TODO：查看时间范围、统计粒度、销量单位和样例记录
for df in dfs:
    print(f"字段: {df.columns.tolist()}")
    print(f"时间范围: {df['time'].min()} to {df['time'].max()}" if 'time' in df.columns else f"Year Range: {df['year'].min()} to {df['year'].max()}")
    print(f"样例记录:\n{df.head()}\n")

字段: ['time', 'total_sales', 'yoy_growth']
时间范围: 2007-01-01 to 2023-09-01
样例记录:
         time  total_sales  yoy_growth
0  2023-09-01    2019445.0     -0.1125
1  2023-08-01    1922495.0     -0.1339
2  2023-07-01    1781580.0     -0.1620
3  2023-06-01    1894250.0     -0.1254
4  2023-05-01    1685966.0      0.0691

字段: ['year', 'month', 'model', 'manufacturer', 'sales', 'price']
Year Range: 2015.0 to 2023.0
样例记录:
     year  month        model manufacturer    sales        price
0  2022.0    1.0           轩逸         东风日产  61170.0   9.98-17.49
1  2022.0    1.0           朗逸         上汽大众  45524.0   9.40-15.19
2  2022.0    1.0     宏光MINIEV       上汽通用五菱  37048.0    3.28-9.99
3  2022.0    1.0         哈弗H6         长城汽车  35570.0   9.89-15.70
4  2022.0    1.0  长安CS75 PLUS         长安汽车  33590.0  11.79-15.49

字段: ['year', 'month', 'model', 'manufacturer', 'sales', 'price']
Year Range: 2015.0 to 2023.0
样例记录:
     year  month     model manufacturer    sales          price
0  2023.0    1.0  宋PLUS新能源     

记录数据文件来源、版本和最新更新时间<br/>
数据文档
【中国电动汽车销售数据】2015~2023年各厂商各车型
<br/>
一、数据介绍
<br/>
数据范围：年份，月份
<br/>
数据年份：2015~2023/11月
<br/>
样本数量：三张表
<br/>
三、数据概览
<br/>
excel格式

---
# 数据质量检查

在正式分析前，检查缺失值、重复记录、年月是否连续、销量是否存在明显异常，以及厂商、品牌和车型字段能否正确对应。这个步骤能帮助避免后续指标因数据问题产生误读。

In [21]:
# TODO：检查字段类型、缺失值和重复记录
_ = duckdb.connect(db)

for df in dfs:
    print(f"检查表: {df.columns.tolist()}")
    print("字段类型:")
    print(df.dtypes)
    print("\n缺失值统计:")
    print(df.isnull().sum())
    print("\n重复记录统计:")
    print(df.duplicated().sum())
    print("\n")

_.close()

检查表: ['time', 'total_sales', 'yoy_growth']
字段类型:
time            object
total_sales    float64
yoy_growth     float64
dtype: object

缺失值统计:
time            0
total_sales     0
yoy_growth     12
dtype: int64

重复记录统计:
0


检查表: ['year', 'month', 'model', 'manufacturer', 'sales', 'price']
字段类型:
year            float64
month           float64
model               str
manufacturer        str
sales           float64
price               str
dtype: object

缺失值统计:
year            0
month           0
model           0
manufacturer    0
sales           0
price           0
dtype: int64

重复记录统计:
42


检查表: ['year', 'month', 'model', 'manufacturer', 'sales', 'price']
字段类型:
year            float64
month           float64
model               str
manufacturer        str
sales           float64
price               str
dtype: object

缺失值统计:
year            0
month           0
model           0
manufacturer    0
sales           0
price           0
dtype: int64

重复记录统计:
167




In [24]:
# TODO：检查年月连续性、缺失月份和数据截止月份
for df in dfs:
    if 'year' in df.columns and 'month' in df.columns:
        df['date'] = pd.to_datetime(df[['year', 'month']].assign(day=1))
        date_range = pd.date_range(start=df['date'].min(), end=df['date'].max(), freq='MS')
        missing_dates = date_range.difference(df['date'])
        print(f"检查表: {df.columns.tolist()}")
        print(f"数据截止月份: {df['date'].max()}")
        print(f"缺失月份: {missing_dates}\n")

检查表: ['year', 'month', 'model', 'manufacturer', 'sales', 'price', 'date']
数据截止月份: 2023-09-01 00:00:00
缺失月份: DatetimeIndex([], dtype='datetime64[us]', freq='MS')

检查表: ['year', 'month', 'model', 'manufacturer', 'sales', 'price', 'date']
数据截止月份: 2023-10-01 00:00:00
缺失月份: DatetimeIndex([], dtype='datetime64[us]', freq='MS')



In [26]:
# TODO：检查负销量
for df in dfs:
    if 'sales' in df.columns:
        negative_sales = df[df['sales'] < 0]
        print(f"检查表: {df.columns.tolist()}")
        print(f"负销量记录:\n{negative_sales}\n")

检查表: ['year', 'month', 'model', 'manufacturer', 'sales', 'price', 'date']
负销量记录:
Empty DataFrame
Columns: [year, month, model, manufacturer, sales, price, date]
Index: []

检查表: ['year', 'month', 'model', 'manufacturer', 'sales', 'price', 'date']
负销量记录:
Empty DataFrame
Columns: [year, month, model, manufacturer, sales, price, date]
Index: []



### 数据质量记录
<!-- 建议记录：发现了什么问题、如何处理、处理后是否仍有影响。例如：某月份缺少记录，因此不参与环比计算。 -->

---
# 数据清洗

根据前面的检查结果处理缺失值、重复值和明显异常记录，并保留必要的清洗说明。清洗的目标是让数据可以用于后续的月度比较，而不是为了让数据看起来更“漂亮”。

In [37]:
# TODO：清洗数据
for i,df in enumerate(dfs):
    df = df.drop_duplicates()  # 删除重复记录
    df = df.dropna()  # 删除缺失值记录
    df = df[df['sales'] >= 0] if 'sales' in df.columns else df  # 删除负销量记录
    df = df.reset_index(drop=True)  # 重置索引
    dfs[i] = df

In [38]:
# TODO: 检查数据清洗质量
for df in dfs:
    print(f"检查表: {df.columns.tolist()}")
    print("字段类型:")
    print(df.dtypes)
    print("\n缺失值统计:")
    print(df.isnull().sum())
    print("\n重复记录统计:")
    print(df.duplicated().sum())
    print("\n负销量记录统计:")
    if 'sales' in df.columns:
        negative_sales = df[df['sales'] < 0]
        print(f"负销量记录:\n{negative_sales}\n")

检查表: ['time', 'total_sales', 'yoy_growth']
字段类型:
time            object
total_sales    float64
yoy_growth     float64
dtype: object

缺失值统计:
time           0
total_sales    0
yoy_growth     0
dtype: int64

重复记录统计:
0

负销量记录统计:
检查表: ['year', 'month', 'model', 'manufacturer', 'sales', 'price', 'date']
字段类型:
year                   float64
month                  float64
model                      str
manufacturer               str
sales                  float64
price                      str
date            datetime64[us]
dtype: object

缺失值统计:
year            0
month           0
model           0
manufacturer    0
sales           0
price           0
date            0
dtype: int64

重复记录统计:
0

负销量记录统计:
负销量记录:
Empty DataFrame
Columns: [year, month, model, manufacturer, sales, price, date]
Index: []

检查表: ['year', 'month', 'model', 'manufacturer', 'sales', 'price', 'date']
字段类型:
year                   float64
month                  float64
model                      str
manufacturer             

---
# 宏观数据分析
这一部分从市场整体出发，比较汽车总销量与新能源汽车销量，并初步了解新能源汽车在汽车市场中的占比、品牌份额和品牌排名变化。

In [73]:
# 按月份的整体汽车销量和新能源汽车销量趋势图
fig = go.Figure()
v_sales = v_df.groupby(['year', 'month'])['sales'].sum().reset_index()
ev_sales = ev_df.groupby(['year', 'month'])['sales'].sum().reset_index()

fig.add_trace(go.Scatter(x=pd.to_datetime(v_sales[['year', 'month']].assign(day=1)), y=v_sales['sales'], mode='lines', name='整体汽车销量'))
fig.add_trace(go.Scatter(x=pd.to_datetime(ev_sales[['year', 'month']].assign(day=1)), y=ev_sales['sales'], mode='lines', name='新能源汽车销量'))

#绘制线性回归直线
fig.add_trace(go.Scatter(x=pd.to_datetime(v_sales[['year', 'month']].assign(day=1)), y=np.poly1d(np.polyfit(pd.to_datetime(v_sales[['year', 'month']].assign(day=1)).astype(int), v_sales['sales'], 1))(pd.to_datetime(v_sales[['year', 'month']].assign(day=1)).astype(int)), mode='lines', name='整体汽车销量趋势线'))
fig.add_trace(go.Scatter(x=pd.to_datetime(ev_sales[['year', 'month']].assign(day=1)), y=np.poly1d(np.polyfit(pd.to_datetime(ev_sales[['year', 'month']].assign(day=1)).astype(int), ev_sales['sales'], 1))(pd.to_datetime(ev_sales[['year', 'month']].assign(day=1)).astype(int)), mode='lines', name='新能源汽车销量趋势线'))

fig.show()

In [72]:
# 按年份的整体汽车销量和新能源汽车销量趋势图
fig = go.Figure()
v_sales_year = v_df.groupby(['year'])['sales'].sum().reset_index()
ev_sales_year = ev_df.groupby(['year'])['sales'].sum().reset_index()

fig.add_trace(go.Scatter(x=v_sales_year['year'], y=v_sales_year['sales'], mode='lines+markers', name='整体汽车销量'))
fig.add_trace(go.Scatter(x=ev_sales_year['year'], y=ev_sales_year['sales'], mode='lines+markers', name='新能源汽车销量'))

#绘制电车占比柱状图
# fig.add_trace(go.Bar(x=ev_sales_year['year'], y=ev_sales_year['sales']/v_sales_year['sales'], name='新能源汽车占比', yaxis='y2'))

fig.show()

---
# 按品牌的研究分析

这一部分按品牌比较当月销量、同比、环比、市场份额和排名变化，并关注重点品牌最近12个月的走势。

建议图表：品牌销量横向排名图、份额趋势图、排名变化图和同比—环比四象限图。可以设置分析月份、品牌、TOP N和时间范围等筛选条件。

In [ ]:
# TODO：计算品牌销量、同比/环比、市场份额、排名及排名变化

In [ ]:
# TODO：使用Plotly绘制品牌排名、份额趋势、排名变化和同比—环比四象限图

### 数据观察
<!-- 写清月份、品牌、数值和比较基期。例如：某品牌在某月销量排名上升X位。 -->

### 可能原因
<!-- 可以提出上新、价格调整或促销等可能解释，但不要直接写成确定原因。 -->

### 待验证假设
<!-- 写下下一步想验证的问题，以及需要补充什么数据。 -->

---
# 按车型的研究分析

这一部分按车型观察销量排名、品牌内车型数量和TOP1/TOP3车型贡献，了解品牌销量是否依赖少数热门车型。

建议图表：车型销量排行榜、品牌车型结构堆叠图、车型贡献率图和重点车型月度趋势图。

In [ ]:
# TODO：计算车型销量排名、车型数量、TOP1/TOP3贡献率及车型状态

In [ ]:
# TODO：使用Plotly绘制车型排名、品牌车型结构、贡献率和重点车型趋势

### 数据观察
<!-- 写清月份、车型或品牌、数值和比较基期。 -->

### 可能原因
<!-- 可以提出新品上市、车型换代等可能解释，但不要直接写成确定原因。 -->

### 待验证假设
<!-- 写下下一步想验证的问题，以及需要补充什么数据。 -->

---
# 异常波动监控

这一部分按照前面设定的规则，找出销量、份额或排名变化较大的品牌和车型，并整理成便于复查的异常清单。

异常分组：销量明显增长、销量明显下降、份额明显变化、排名明显变化、新增记录和低基数波动。低基数对象单独标注，避免小销量变化被放大解读。

建议图表：异常月份时间线、异常对象排行榜、月度销量热力图和异常明细表。异常只是值得关注的信号，后续仍需查看更详细的资料。

In [ ]:
# TODO：应用固定阈值和低基数规则，生成品牌与车型异常清单

In [ ]:
# TODO：使用Plotly绘制异常时间线、异常排名、销量热力图和异常明细表

### 数据观察
<!-- 写清异常发生的月份、对象、触发规则和具体数值。 -->

### 可能原因
<!-- 说明可能需要查看哪些外部信息，但不要仅凭销量变化确定原因。 -->

### 待验证假设
<!-- 写下需要补充的数据，以及准备如何验证。 -->

---
# Plotly可筛选经营看板

## 看板布局

1. 顶部：分析月份及核心KPI。
2. 左侧：最近12个月销量、同比和环比趋势。
3. 右侧：品牌销量排名及市场份额。
4. 下方：车型结构和车型贡献。
5. 底部：异常清单与异常趋势。

## 交互设计

- Plotly日期范围滑块。
- 品牌下拉选择。
- 销量、同比、环比和份额指标切换。
- TOP N切换。
- 异常类型筛选。
- Notebook参数单元格统一控制分析月份。

图表使用白色主题、固定品牌配色，并写清单位和比较月份。标题以描述图表内容为主，不提前写出结论。

In [ ]:
# TODO：根据分析月份和筛选参数准备Plotly看板数据

In [ ]:
# TODO：使用Plotly组装可筛选的月度经营看板

---
# 本月经营摘要与待跟进事项

这一页用于面试展示时快速说明“本月发生了什么、我从数据中看到了什么、下一步还想验证什么”。所有结论应以本Notebook前面的图表和指标为依据。

## 本月核心表现

| 指标 | 本月结果 |
|---|---|
| 当月销量 | 待填写 |
| 同比 | 待填写 |
| 环比 | 待填写 |
| 新能源渗透率 | 待填写 |
| 市场份额最高品牌 | 待填写（写明份额） |
| 变化最大品牌 | 待填写（写明变化指标） |
| 变化最大车型 | 待填写（写明变化指标） |

## 数据观察
<!-- 用2—3句话总结最重要的数据变化，并写清对象、月份和数值。 -->

## 可能原因
<!-- 用“可能与……有关”描述原因，不把推测写成结论。 -->

## 待验证假设

| 想验证的问题 | 需要补充的数据 | 我准备怎么验证 | 优先级 |
|---|---|---|---|
| 待填写 | 待填写 | 待填写 | 待填写 |

## 待跟进事项

| 下一步要做什么 | 关联的变化 | 需要查看的数据/资料 | 完成情况 |
|---|---|---|---|
| 待填写 | 待填写 | 待填写 | 待填写 |

In [ ]:
# TODO：根据选定月份的已验证指标生成本月核心表现摘要所需数据

---
# 数据限制与后续方向

- 记录实际数据截止月份、缺失字段和清洗处理。
- 同比和环比可能受到季节性、春节月份错位及低基数影响。
- 销量数据可以描述波动，但不能单独解释波动原因。
- 原因验证通常需要价格、渠道、促销、产能、库存、政策和舆情等补充数据。
- 目前的分析重点是使用Pandas和Plotly完成Notebook内的月度监控；如有需要，月度指标表和异常清单也可以继续用于Power BI或Tableau。
- 后续会从头运行Notebook，核对指标、筛选状态、图表标签和最终摘要。